In [ ]:
%%capture
import os
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
reports_folder = Path(os.environ["INTECOMM_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from tabulate import tabulate
from edc_constants.constants import NO
from edc_pdutils.dataframes import get_crf, get_subject_visit
from edc_visit_tracking.constants import MISSED_VISIT
from intecomm_analytics.utils import get_formatted_rows_by_country_single
from edc_constants.constants import OTHER

In [ ]:
df_main_original = get_df_main_1858(None)
df_main = df_main_original.copy()


In [ ]:
df_visit = get_subject_visit("intecomm_subject.subjectvisit")
df_visit = df_visit.merge(df_main[["subject_identifier", "assignment", "hiv", "htn", "dm", "baseline_datetime", "endline_datetime", "onstudy_days", "country"]], on="subject_identifier", how="left")
df_visit = df_visit[df_visit.reason!=MISSED_VISIT].copy()
df_visit.reset_index(drop=True, inplace=True)

In [ ]:
df = df_visit.groupby(by=["subject_identifier", "assignment", "hiv", "htn", "dm", "country"])["visit_code"].count().to_frame()
df.rename(columns={"visit_code": "freq"}, inplace=True)
df.reset_index(inplace=True)
df["freq"].describe()


In [ ]:
def get_formatted_rows(*args):
    dct = get_formatted_rows_by_country_single(*args)
    del dct["Timepoint"]
    return dct


In [ ]:
table = {'Condition': ['All', '', '']}
table.update({
    'Parameter': ['Visit frequency', '', ''],
    **get_formatted_rows(df, "freq")
})
table_df = pd.DataFrame(table)
all_tables = table_df.copy()


In [ ]:
table = {'Condition': ['DM and HTN', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==0) & (df.htn==1) & (df.dm==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)

In [ ]:
table = {'Condition': ['DM and HIV', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==1) & (df.htn==0) & (df.dm==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)

In [ ]:
table = {'Condition': ['HTN and HIV', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==1) & (df.htn==1) & (df.dm==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)

In [ ]:
table = {'Condition': ['DM and HTN and HIV', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==1) & (df.htn==1) & (df.dm==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['DM only', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==0) & (df.htn==0) & (df.dm==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['HTN only', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==0) & (df.htn==1) & (df.dm==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['HIV only', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==1) & (df.htn==0) & (df.dm==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['without HTN', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.htn==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['with HTN', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.htn==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['without DM', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.dm==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['with DM', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.dm==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['without HIV', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==0)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
table = {'Condition': ['with HIV', '', '']}
table.update({
    'Parameter': ['', '', ''],
    **get_formatted_rows(df[(df.hiv==1)], "freq")
})
table_df = pd.DataFrame(table)
all_tables = pd.concat([all_tables, table_df], ignore_index=True)


In [ ]:
all_tables

In [ ]:
all_tab = tabulate(all_tables, headers='keys', tablefmt='grid')

path = analysis_folder / 'visits.csv'
all_tables.to_csv(path_or_buf=path, index=False)


In [ ]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(df[(df.htn==1)]['freq'], df[(df.htn==0)]['freq'])

In [ ]:
t_stat, p_value

In [ ]:
t_stat, p_value = ttest_ind(df[() & (df.htn==1)]['freq'], df[(df.htn==0)]['freq'])


In [ ]:
df_main = get_df_main_1858(None)

In [ ]:
df_main.appt_type

In [ ]:
df_location_update = get_crf(
    "intecomm_subject.locationupdate",
    subject_visit_model="intecomm_subject.subjectvisit",
    read_verbose=False,
)


In [ ]:
df_location_update = df_location_update[df_location_update.visit_code<1120.0].copy()


In [ ]:

df_location_update["location"] = df["location"].map({"community":"a", "clinic": "b", OTHER: OTHER})

In [ ]:
df_location_update["location"] = df["location"].map({"community":"a", "clinic": "b", OTHER: OTHER})
df_location_update.rename(columns={"next_location": "returning"}, inplace=True)
df_location_update = df_location_update.merge(df_main[["subject_identifier", "assignment", "hiv", "dm", "htn", "country"]], how="left", on="subject_identifier")



In [ ]:
def get_direction(s):
    if s.location == OTHER:
        return OTHER
    return f"{s.location}->{s.assignment}"
df_location_update["direction"] = df_location_update.apply(get_direction, axis=1)


In [ ]:
df_location_update.direction.value_counts()

In [ ]:
df_location_update = df_location_update[df_location_update.direction.isin(["b->a", OTHER])].copy()
df_location_update.groupby(by=["direction", "returning"]).size()

In [ ]:
df1 = df[(df.location_assign != df.assignment) & (df.visit_code <= 1060.0)][["subject_identifier", "country", "hiv", "dm", "htn", "visit_code", "direction", "assignment", "returning", "comments"]].copy()
df1.sort_values(by=["subject_identifier", "visit_code"], ascending=[True, True], inplace=True)
df1

In [ ]:
df1.groupby(by=["hiv", "dm", "htn"]).size()

In [ ]:
df1.next_location.value_counts()

In [ ]:
df1[df1.next_location==NO]

In [ ]:
df_main[df_main.subject_identifier=="107-101-0081-8"]

In [ ]:
df_visit = get_subject_visit("intecomm_subject.subjectvisit")

In [ ]:
df = pd.merge(df_visit[df_visit.subject_identifier.isin(df_location_update.subject_identifier)][["subject_identifier", "visit_code", "reason"]], df_location_update[["subject_identifier", "country", "hiv", "dm", "htn", "visit_code", "direction", "assignment", "returning", "comments"]], on=["subject_identifier", "visit_code"], how="left")

In [ ]:
df

In [ ]:
df_visit[df_visit.reason=="missed"].groupby(by=["subject_identifier"]).size()


In [ ]:
df_visit

In [ ]:
df_visit = df_visit.merge(df_main[["subject_identifier", "assignment"]], on=["subject_identifier"], how="left")

In [ ]:
df_visit[(df_visit.visit_code>1000.0) & (df_visit.visit_code<1120.0)].groupby(by=["appt_type", "assignment"]).size()


In [ ]:
df_visit[(df_visit.visit_code>1000.0) & (df_visit.visit_code<1120.0) & (df_visit.appt_type == "clinic") & (df_visit.assignment == "a")][["subject_identifier", "visit_code", "appt_type", "assignment" ]]

In [ ]:
df_location_update = get_crf(
    "intecomm_subject.locationupdate",
    subject_visit_model="intecomm_subject.subjectvisit",
    read_verbose=False,
)
df_location_update = df_location_update[(df_location_update.visit_code>1000.0) & (df_location_update.visit_code<1120.0)].copy()
df_location_update = df_location_update.merge(df_main[["subject_identifier", "assignment", "country", "hiv", "dm", "htn"]], on=["subject_identifier"], how="left")
df_location_update["location"] = df_location_update["location"].map({"community":"a", "clinic": "b", OTHER: OTHER})
df_location_update.rename(columns={"next_location": "returning"}, inplace=True)

def get_direction(s):
    if s.location == OTHER:
        return OTHER
    return f"{s.assignment}->{s.location}"

def is_before_6m(s):
    if (s.report_datetime - s.baseline_datetime).days < 182:
        return 1
    return 0

df_location_update["direction"] = df_location_update.apply(get_direction, axis=1)
df_location_update["<182"] = df_location_update.apply(is_before_6m, axis=1)
df_location_update = df_location_update[~df_location_update.direction.isin(["a->a", "b->b", "OTHER"])].copy()



In [ ]:
df_location_update.sort_values(by=["subject_identifier", "visit_code"], ascending=[True, True], inplace=True)

In [ ]:
df_location_update[["subject_identifier", "report_datetime", "<182", "country", "hiv", "dm", "htn", "visit_code", "direction", "assignment", "returning", "comments"]]

In [ ]:
df_location_update[df_location_update["<182"]==1].groupby(by=["subject_identifier"]).nunique()[["visit_code"]]>1

In [ ]:
df_location_update